# Module 13 — Guardrails and budgets

**THE ONE IDEA:** five guards, each aimed at a specific failure from module 12, wrapped
around the same naive loop. **The pass condition is that all five scripts get caught.**

| guard | catches | how |
|---|---|---|
| 1. rescue-from-content | FM1 | regex the JSON back out of `content` |
| 2. no-progress stop | FM2, FM3 | no NEW information this turn -> stop |
| 3. duplicate detection | FM4 | `(tool, args)` seen before -> serve the cache |
| 4. Pydantic arg validation | FM5 | unknown tool or bad args -> structured error |
| 5. token / iteration budget | all | hard ceiling, then **abort and escalate** |

Every guard returns a **structured error to the model**, never an exception — a wrong
tool name is recoverable next turn, and raising throws that chance away.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import json, re
from pydantic import BaseModel, ValidationError
from _fake_model import FakeModel, tool_turn, text_turn, raw_turn
from _tools import REGISTRY, run_tool

class SearchPolicy(BaseModel): query: str
class Calculate(BaseModel):    expression: str
SCHEMAS = {"search_policy": SearchPolicy, "calculate": Calculate}

# NOTE \s* WITH the backslash. 8.agents/02 had `\{s*"name"` = zero-or-more letter 's',
# which matched compact JSON and silently missed pretty-printed JSON. Fixed there too.
RESCUE_RE = re.compile(r'\{\s*"name"\s*:\s*"(\w+)"\s*,\s*"arguments"\s*:\s*(\{.*?\})', re.S)

def validate(name, args):
    if name not in REGISTRY:
        return None, f"ERROR: unknown tool '{name}'. Available: {', '.join(REGISTRY)}"
    if name not in SCHEMAS: return args, None
    try: return SCHEMAS[name](**args).model_dump(), None
    except ValidationError as e: return None, f"ERROR: bad args for '{name}': {e.errors()[0]['loc']}"

def _fake_call(name, args_json):     # mimic the SDK tool-call object for a rescued call
    return type("T", (), {"id": "rescued",
                          "function": type("F", (), {"name": name, "arguments": args_json})()})()

## The guarded loop

In [ ]:
def guarded(fake, max_steps=8, token_budget=4000, verbose=True):
    messages = [{"role": "user", "content": "What is the year-2 ERC on 250000?"}]
    seen, facts, tokens, nudged = set(), set(), 0, False
    for step in range(1, max_steps + 1):
        r = fake.create(messages=messages)
        msg = r.choices[0].message
        tokens += r.usage.prompt_tokens + r.usage.completion_tokens
        if tokens > token_budget:                                        # GUARD 5
            return "ABORT: token budget exceeded — escalate to a human", step

        calls = msg.tool_calls
        if not calls and msg.content and (m := RESCUE_RE.search(msg.content)):
            if verbose: print(f"  step {step}: GUARD1 rescued {m.group(1)} from content")
            calls = [_fake_call(m.group(1), m.group(2))]                  # GUARD 1
        if not calls:
            if not facts:                                                 # GUARD 2a
                if nudged:
                    return "ABORT: described a plan, never acted — escalate", step
                nudged = True
                if verbose: print(f"  step {step}: GUARD2 plan-not-action — nudging")
                messages.append({"role": "user", "content":
                                 "Do not describe the plan. Call the tool NOW."})
                continue
            return msg.content, step

        progressed = False
        for tc in calls:
            try: args = json.loads(tc.function.arguments)
            except json.JSONDecodeError: args = {}
            args, err = validate(tc.function.name, args)                  # GUARD 4
            if err:
                if verbose: print(f"  step {step}: GUARD4 {err[:60]}")
                messages.append({"role": "user", "content": err}); progressed = True; continue
            sig = (tc.function.name, json.dumps(args, sort_keys=True))
            if sig in seen:                                               # GUARD 3
                if verbose: print(f"  step {step}: GUARD3 duplicate — serving cache")
                continue
            seen.add(sig); out = run_tool(tc.function.name, args)
            if out not in facts: facts.add(out); progressed = True
            if verbose: print(f"  step {step}: ran {tc.function.name} -> {out[:46]}")

        if not progressed:                                                # GUARD 2b
            return "STOP: no new information this turn", step
    return "ABORT: iteration cap — escalate to a human", max_steps

## Re-run all five module-12 scripts

In [ ]:
S = {"FM1": [raw_turn('Sure. {"name": "search_policy", "arguments": {"query": "erc"}}',
                      None, "stop"), text_turn("Year-2 ERC is 10000.")],
     "FM2": [text_turn("My plan: search the ERC policy, then calculate 4%.")],
     "FM3": [tool_turn("search_policy", {"query": "erc"}, "c1"),
             tool_turn("calculate", {"expression": "250000*0.04"}, "c2"),
             tool_turn("search_policy", {"query": "erc"}, "c3")],
     "FM4": [tool_turn("search_policy", {"query": "erc"}, "c1")],
     "FM5": [tool_turn("lookup_acct", {"query": "erc"}, "c1"),
             tool_turn("calculate", {"principal_rate_pct": "250000*0.04"}, "c2"),
             text_turn("Year-2 ERC is 10000.")]}

results = {}
for tag, script in S.items():
    print(f"\n--- {tag} ---")
    results[tag] = guarded(FakeModel(script))
    print(f"  => {results[tag][0]}  (step {results[tag][1]})")

## Pass condition

In [ ]:
LABEL = {"FM1": "guard 1 rescue-from-content", "FM2": "guard 2a nudge, then abort",
         "FM3": "guard 2b + 3", "FM4": "guard 3 duplicate detect",
         "FM5": "guard 4, then 2a abort"}
print(f"{'mode':5} {'caught by':28} {'steps':>5}\n" + "-" * 42)
for tag, (out, steps) in results.items():
    print(f"{tag:5} {LABEL[tag]:28} {steps:5}")

print("""
All five caught. In module 12, FM4 burned all 8 steps and FM1/FM2 both returned
a confident non-answer.

LESSON - each guard maps to a failure you can REPRODUCE. That mapping is the
deliverable. A guard with no matching script is untested code, and an untested
guard is worse than none: it buys confidence you have not earned.

FM2 is the instructive one. The FIRST draft of this notebook CLAIMED to catch it
and did not - the loop returned the prose plan as an answer, exactly like the
unguarded version. Only re-running module 12's scripts exposed it. That is the
whole argument for scripting failures.

The last defence ABORTS AND ESCALATES rather than returning a partial answer as
though it were complete. Module 14 adds the trail that makes escalation actionable.
""")

---

**Next:** `14_audit_log_and_hitl.ipynb`